# Configs B, B′, C-noaug, C (ResNet-18) + submission

All ResNet-18 experiments for Dogs vs. Cats, plus the final `submission.csv`.

| Config | What it is | Train augmentation |
|---|---|---|
| **B** | Frozen ResNet-18 features → SVM | No |
| **B′** | Frozen ResNet-18 features → linear layer | No |
| **C-noaug** | ResNet-18, layer4 + linear layer fine-tuned | No |
| **C** | Same as C-noaug, **final model** | Yes |

Comparison chain for Part f (each step changes one thing):
- B to B' changes the classifer
- B' to C no aug unfreezes the last block of ResNet-18 (layer4) and allows learning capabilities so it can adapt to cats vs dogs
- C no aug to C adds augmentation (each training picture is randomly rotated, flipped, cropped and colour-adjusted every time it is shown)

**How to run:** Kernel → Restart, then Run All.

- `FULL_RUN = False`: 2,000 train / 500 val images, for testing on your own laptop.
- `FULL_RUN = True`: all 20,000 / 5,000 images. **Only the trainer sets this.**
- The LR sweep (section 2) **always** uses the small subset, whatever `FULL_RUN` is.

Outputs (paths fixed by `common.py`):

- `outputs/results_<config>_seed42.csv` for `sweep-lr1e-3`, `sweep-lr1e-4`, `sweep-lr1e-5`, `B`, `Bp`, `C-noaug`, `C`
- `outputs/preds_C_seed42.csv` (per-image val predictions, for Part e)
- `checkpoints/linear_Bp_seed42.pth`, `checkpoints/resnet18_C-noaug_seed42.pth`, `checkpoints/resnet18_C_seed42.pth` (shared Drive, not GitHub)
- `submission.csv` (full run only; a subset run writes `submission_SUBSET_DO_NOT_SUBMIT.csv` instead)

⚠️ Testing on your own machine: **do not commit** `outputs/`, `checkpoints/` or any submission file. Only the trainer commits outputs.

In [ ]:
FULL_RUN = True   # trainer only: set True for the final run

In [ ]:
import time

import pandas as pd
import torch
import torch.nn as nn
from sklearn.svm import SVC
from torch.utils.data import TensorDataset

from common import *

set_seed()
print("device:", DEVICE, "| FULL_RUN:", FULL_RUN)

## 1. Dataset (Part a)

Image counts per split and class. Train/val come with labels (folder names: cat = 0, dog = 1);
the 500 test images have no labels.

In [ ]:
counts = describe_dataset()
print(counts.to_string(index=False))

## Shared training settings

Same for **every** trained config in this notebook and in `01_configA_cnn.ipynb`, so differences
in accuracy come from the model, not from training longer.

| Setting | Value | Reason |
|---|---|---|
| Optimizer | Adam | Adaptive step size per parameter (Week 11) |
| Max epochs | 15 | Same budget for A, B′, C-noaug, C |
| Scheduler | StepLR, ×0.5 every 3 epochs | Same decay for all; halves the LR 4 times in 15 epochs |
| Checkpoint kept | Best val accuracy | "Up to 15 epochs", same rule for every config |
| Loss | CrossEntropyLoss | Standard loss for class logits |
| Batch size | 32 | Fixed in `common.py` |

Starting LR differs **by design**: B′ trains a brand-new layer (1e-3), while C and C-noaug adjust
pretrained weights and need smaller steps or the ImageNet knowledge is destroyed (Week 8).
The C learning rate is chosen by the sweep in section 2.

In [ ]:
EPOCHS = 15
STEP_SIZE = 3
GAMMA = 0.5
LR_BP = 1e-3        # B′: new linear layer, random start
LR_C_DEFAULT = 1e-4  # C and C-noaug: standard fine-tuning LR, kept unless the sweep CLEARLY beats it
SWEEP_MARGIN = 0.01  # "clearly" = at least 1 percentage point better val accuracy (5 of 500 subset images)
LR_C = None          # None = decide from the sweep (rule above); or set a number to override

SWEEP_LRS = {"sweep-lr1e-3": 1e-3, "sweep-lr1e-4": 1e-4, "sweep-lr1e-5": 1e-5}
SWEEP_EPOCHS = 3

criterion = nn.CrossEntropyLoss()


def make_optimizer(model, lr):
    """Adam on the trainable parameters only + the shared StepLR schedule."""
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)
    return optimizer, scheduler


def release_gpu():
    """Return freed GPU memory between configs (the trainer's GPU may only have 4 GB).
    Call after `del model, optimizer` so the old model is really gone."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 2. Learning-rate sweep for Config C (Step 4 / Part c)

Config C's setup trained for a few epochs at three learning rates, **always on the small subset**
(2,000 / 500 images) so it takes minutes. All three run here, on the same machine, so they are comparable.
**Decision rule:** keep 1e-4 unless another LR beats it by at least 1 percentage point
(the subset has only 500 val images, so smaller gaps are just a few images, i.e. noise).
The chosen LR is used for both C-noaug and C, so they still differ only in augmentation.

This gives evidence for the LR choice in the report instead of "1e-4 is standard".
Limitation to state: a short run on a subset is a guide, not a guarantee for the full run.

In [ ]:
sweep_train, sweep_val = get_dataloaders(full_run=False)

sweep_rows = []
for config, lr in SWEEP_LRS.items():
    set_seed()   # same starting weights for every LR
    model = get_resnet18(mode="finetune")
    optimizer, scheduler = make_optimizer(model, lr)
    hist = fit(model, config, sweep_train, sweep_val, optimizer, criterion,
               epochs=SWEEP_EPOCHS, scheduler=scheduler, keep_bn_train=("layer4",))
    sweep_rows.append({"config": config, "lr": lr, "best_val_acc": hist.eval_acc.max(),
                       "final_train_loss": hist.train_loss.iloc[-1]})
    del model, optimizer, scheduler
    release_gpu()

sweep = pd.DataFrame(sweep_rows)
best_sweep = sweep.loc[sweep.best_val_acc.idxmax()]
print(sweep.to_string(index=False))

default_acc = sweep.loc[sweep.lr == LR_C_DEFAULT, "best_val_acc"].iloc[0]
if LR_C is not None:
    reason = f"set manually (sweep best was {best_sweep.lr:g})"
elif best_sweep.lr != LR_C_DEFAULT and best_sweep.best_val_acc >= default_acc + SWEEP_MARGIN:
    LR_C = float(best_sweep.lr)
    reason = f"sweep: beats {LR_C_DEFAULT:g} by {best_sweep.best_val_acc - default_acc:.3f} (>= {SWEEP_MARGIN})"
else:
    LR_C = LR_C_DEFAULT
    reason = f"default kept: no LR beats {LR_C_DEFAULT:g} by >= {SWEEP_MARGIN} val accuracy"
print(f"\nLR for C and C-noaug = {LR_C:g} ({reason})")

## 3. Shared fixed features for B and B′

The backbone is frozen and its BatchNorm is fixed, so **without augmentation** every image always gives
the same 512 numbers. We compute them **once** and both B and B′ learn from exactly the same features.
This is mathematically the same as running the frozen network every epoch, but B′'s 15 epochs take
seconds instead of 15 full passes.

⚠️ Must use `augment=False`: with augmentation, each image would be a random distorted version.

In [ ]:
feat_train_loader, feat_val_loader = get_dataloaders(FULL_RUN, augment=False)

set_seed()
backbone = get_resnet18(mode="features")   # frozen, fc removed -> 512-dim output
t0 = time.time()
X_train, y_train = extract_features(backbone, feat_train_loader)
X_val, y_val = extract_features(backbone, feat_val_loader)
print(f"features: train {X_train.shape}, val {X_val.shape} | {time.time() - t0:.0f}s")
del backbone
release_gpu()

## 4. Config B: SVM on frozen features (Week 6)

RBF-kernel SVM, `C=1.0`, `gamma='scale'` (sklearn default). The SVM sees the raw 512 features, exactly
what B′ sees, so B vs B′ compares only the classifier. No epochs: the SVM is fitted once.

In [ ]:
t0 = time.time()
svm = SVC(kernel="rbf", C=1.0, gamma="scale")
svm.fit(X_train, y_train)
acc_B = svm.score(X_val, y_val)
print(f"Config B (SVM) val accuracy {acc_B:.4f} | fit + score {time.time() - t0:.0f}s")
save_result_row("B", acc_B, seconds=round(time.time() - t0, 1))

## 5. Config B′: linear layer on the same frozen features

`nn.Linear(512, 2)` is exactly ResNet-18's final layer, trained on the saved features.
Same max epochs and schedule as every other config; LR 1e-3 because the layer starts from random weights.

In [ ]:
bp_train = make_loader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), shuffle=True)
bp_val = make_loader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), shuffle=False)

set_seed()
linear = nn.Linear(512, 2).to(DEVICE)
optimizer, scheduler = make_optimizer(linear, LR_BP)
hist_bp = fit(linear, "Bp", bp_train, bp_val, optimizer, criterion,
              epochs=EPOCHS, scheduler=scheduler, model_name="linear")
del linear, optimizer, scheduler
release_gpu()

## 6. Config C-noaug: fine-tune layer4, no augmentation

Same as C except the training images are not augmented (val/test-style resize + centre crop only).
layer4 and the new final layer learn; everything else stays frozen, including its BatchNorm statistics
(`keep_bn_train=("layer4",)`).

In [ ]:
train_noaug, val_loader = get_dataloaders(FULL_RUN, augment=False)

set_seed()
model = get_resnet18(mode="finetune")
optimizer, scheduler = make_optimizer(model, LR_C)
hist_noaug = fit(model, "C-noaug", train_noaug, val_loader, optimizer, criterion,
                 epochs=EPOCHS, scheduler=scheduler, keep_bn_train=("layer4",))
del model, optimizer, scheduler
release_gpu()

## 7. Config C: fine-tune layer4, with augmentation (final model)

Identical to C-noaug except the shared training augmentation (rotation, flip, crop/scale, colour jitter)
is switched on. Same seed, so both start from the same weights.

In [ ]:
train_aug, val_loader = get_dataloaders(FULL_RUN, augment=True)

set_seed()
model = get_resnet18(mode="finetune")
optimizer, scheduler = make_optimizer(model, LR_C)
hist_c = fit(model, "C", train_aug, val_loader, optimizer, criterion,
             epochs=EPOCHS, scheduler=scheduler, keep_bn_train=("layer4",))

## 8. Val predictions for Config C (Part e)

`fit()` has already loaded C's best epoch back into `model`. One row per val image
(path, true label, predicted label, probability of dog) for the confusion matrix and
correct/incorrect examples in `03_comparison.ipynb`.

In [ ]:
preds_c = predict_val(model, val_loader, "C")
preds_c.head()

## 9. Submission (Part d)

Predicts the 500 test images with C's best epoch and checks the format
(500 rows, columns `id,label`, ids 1..500, labels only 0/1, same ids as `sampleSubmission.csv`).
A subset run writes a clearly named test file instead, so it can never be mistaken for the real one.

In [ ]:
SUBMISSION_PATH = "submission.csv" if FULL_RUN else "submission_SUBSET_DO_NOT_SUBMIT.csv"
submission = write_submission(model, path=SUBMISSION_PATH)
submission.head(10)

## Quick summary

Sanity check for the trainer. The real comparison, curves and analysis are in `03_comparison.ipynb`.

In [ ]:
summary = pd.DataFrame([
    {"config": "B (SVM)", "best_val_acc": acc_B, "best_epoch": "-"},
    {"config": "B′ (linear)", "best_val_acc": hist_bp.eval_acc.max(), "best_epoch": int(hist_bp.eval_acc.idxmax()) + 1},
    {"config": "C-noaug", "best_val_acc": hist_noaug.eval_acc.max(), "best_epoch": int(hist_noaug.eval_acc.idxmax()) + 1},
    {"config": "C (final)", "best_val_acc": hist_c.eval_acc.max(), "best_epoch": int(hist_c.eval_acc.idxmax()) + 1},
])
print(f"FULL_RUN={FULL_RUN} | LR for C = {LR_C:g} ({reason})")
print(summary.to_string(index=False))